<a href="https://colab.research.google.com/github/AntonDozhdikov/AntonDozhdikov/blob/main/Second_var_LLM_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install gensim lightgbm optuna

In [5]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split  # Импорт функции для разделения данных на тренировочные и тестовые наборы
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, RocCurveDisplay
import gensim.downloader as api
from lightgbm import LGBMClassifier, Dataset
import optuna
from optuna.samplers import TPESampler

# Загрузка данных с принудительным строковым типом
df = pd.read_csv('IMDB.csv', dtype={'review': str, 'sentiment': str})

In [6]:
# Проверка и очистка данных
print(f"Пропуски в данных: {df.isnull().sum()}")
df = df.dropna()

# Исправленная предобработка текста
def preprocess(text):
    return str(text).lower().split()  # Гарантированное строковое преобразование

df['tokens'] = df['review'].apply(preprocess)

Пропуски в данных: review       0
sentiment    0
dtype: int64


In [7]:
# Разделение данных с проверкой типов
X_train, X_test, y_train, y_test = train_test_split(
    df['tokens'],
    df['sentiment'].astype(str),  # Гарантия строкового типа меток
    test_size=0.3,
    random_state=42
)

In [8]:
#  Загрузка предобученной модели Word2Vec
w2v_model = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [9]:
#  Создание TF-IDF векторайзера
tfidf = TfidfVectorizer(max_features=10000)
tfidf.fit([' '.join(tokens) for tokens in X_train])

TfidfVectorizer(max_features=10000)

In [10]:
# Преобразование меток в строковый тип
y_train = y_train.astype(str)
y_test = y_test.astype(str)

In [11]:
#  Векторизация с обработкой числовых значений
def vectorize(tokens):
    try:
        # Фильтрация нестроковых элементов
        valid_tokens = [str(t) for t in tokens if isinstance(t, (str, np.str_))]
        text = ' '.join(valid_tokens)

        weights = tfidf.transform([text]).tocsr()
        vocab = tfidf.vocabulary_

        embeddings = []
        for word in valid_tokens:  # Используем очищенные токены
            if word in w2v_model and word in vocab:
                tfidf_weight = weights[0, vocab[word]]
                emb = w2v_model[word] * tfidf_weight
                embeddings.append(emb.astype(np.float32))

        return np.mean(embeddings, axis=0) if embeddings else np.zeros(300)

    except Exception as e:
        print(f"Ошибка обработки: {str(e)[:100]}")
        return np.zeros(300)

#  Проверка перед векторизацией
print("\nПример токенов:", X_train.iloc[0][:5])
print("Тип первого токена:", type(X_train.iloc[0][0]))



Пример токенов: ['as', 'much', 'as', 'i', 'love']
Тип первого токена: <class 'str'>


In [12]:
!pip install joblib

In [13]:
from joblib import Parallel, delayed

# Добавление конвертации перед параллельной обработкой
X_train_vec = np.array([vectorize(tokens) for tokens in X_train], dtype=np.float32)
X_test_vec = np.array([vectorize(tokens) for tokens in X_test], dtype=np.float32)



In [14]:
X_train =X_train_vec
X_test = X_test_vec

In [15]:
#Обучение базовой модели LightGBM
lgb = LGBMClassifier(random_state = 42)
lgb.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of positive: 17411, number of negative: 17589
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.440233 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 35000, number of used features: 300
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.497457 -> initscore=-0.010172
[LightGBM] [Info] Start training from score -0.010172


LGBMClassifier(random_state=42)

In [18]:
# Получаем предсказания модели на тестовых данных
y_pred = lgb.predict(X_test)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [19]:
cm = confusion_matrix(y_test, y_pred)
print("Матрица ошибок:")
print(cm)

Матрица ошибок:
[[5678 1733]
 [1551 6038]]


In [20]:
cr = classification_report(y_test, y_pred, target_names=['Класс 0', 'Класс 1'], digits=4)
print("\nОтчёт о классификации:")
print(cr)


Отчёт о классификации:
              precision    recall  f1-score   support

     Класс 0     0.7854    0.7662    0.7757      7411
     Класс 1     0.7770    0.7956    0.7862      7589

    accuracy                         0.7811     15000
   macro avg     0.7812    0.7809    0.7809     15000
weighted avg     0.7812    0.7811    0.7810     15000



In [21]:
import optuna
import lightgbm as lgb
from sklearn.metrics import f1_score,  accuracy_score
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix

In [22]:
# Оптимизационная функция для Optuna
def objective(trial):
    # Подбор гиперпараметров
    param = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt', 'dart', 'goss']),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', -1, 20),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }

    # Создание и обучение модели
    model = lgb.LGBMClassifier(**param)
    model.fit(X_train, y_train)

    # Предсказания и оценка
    y_pred =  model.predict(X_test)
    y_prob =  model.predict_proba(X_test)[:, 1]  # Получаем вероятности для положительного класса

    #recall = recall_score(y_test, y_pred)
    #precision = precision_score(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    #roc_auc = roc_auc_score(y_test, y_prob)
    #f1 = f1_score(y_test, y_pred)

    return accuracy

In [23]:
# Создание исследования Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-03-14 17:52:22,874] A new study created in memory with name: no-name-0bd52b84-0224-43a5-ac21-59567160efa0
<ipython-input-22-6a4bc297b20f>:11: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1),
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warn

In [31]:
# Вывод результатов
print('Лучшие параметры:', study.best_params)
print('Лучший accuracy:', study.best_value)

Лучшие параметры: {'boosting_type': 'gbdt', 'num_leaves': 66, 'max_depth': 12, 'learning_rate': 0.07482350097076117, 'n_estimators': 490, 'subsample': 0.5882074711914853, 'colsample_bytree': 0.7829470637144212}
Лучший accuracy: 0.8014666666666667


Лучшие параметры: {'boosting_type': 'gbdt', 'num_leaves': 95, 'max_depth': 18, 'learning_rate': 0.10227810225122058, 'n_estimators': 471, 'subsample': 0.9484584550832317, 'colsample_bytree': 0.977657642971348}
Лучший accuracy: 0.8020666666666667

In [32]:
model_X = lgb.LGBMClassifier(**study.best_params, random_state = 42)

In [33]:
# Обучаем модель на тренировочных данных
model_X.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


LGBMClassifier(colsample_bytree=0.7829470637144212,
               learning_rate=0.07482350097076117, max_depth=12,
               n_estimators=490, num_leaves=66, random_state=42,
               subsample=0.5882074711914853)

In [34]:
# Получаем предсказания модели на тестовых данных
y_pred = model_X.predict(X_test)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [36]:
cm1 = confusion_matrix(y_test, y_pred)
print("Матрица ошибок заоптюненой модели:")
print(cm1)

Матрица ошибок заоптюненой модели:
[[5824 1587]
 [1464 6125]]


In [37]:
cr1 = classification_report(y_test, y_pred, target_names=['Класс 0', 'Класс 1'], digits=4)
print("\nОтчёт о классификации заоптюненой модели:")
print(cr1)


Отчёт о классификации заоптюненой модели:
              precision    recall  f1-score   support

     Класс 0     0.7991    0.7859    0.7924      7411
     Класс 1     0.7942    0.8071    0.8006      7589

    accuracy                         0.7966     15000
   macro avg     0.7967    0.7965    0.7965     15000
weighted avg     0.7966    0.7966    0.7966     15000

